In [2]:
from wgpu import BufferUsage, ShaderStage, BufferBindingType
import wgpu
import numpy as np
import random as rd

1. Les données sur le CPU, avec une version CPU de référence.


In [ ]:
def create_initial_data(size:(int, int), starting_population:int):
    data0 = np.zeros(size, dtype=np.uint32)
    data1 = np.zeros(size, dtype=np.uint32)
    for i in range(starting_population):
        x = rd.randint(0, size[0]-1)
        y = rd.randint(0, size[1]-1)
        data0[x][y] = 1
    return data0,data1
data0, data1= create_initial_data((10, 10), 20)

2. Le canal avec le GPU.

In [ ]:
adapter = wgpu.gpu.request_adapter_sync(power_preference="high-performance")
device = adapter.request_device_sync()

3. Les buffers, y compris le petit buffer de réglages.

In [ ]:
buffer_0 = device.create_buffer_with_data(data=data0, usage=BufferUsage.STORAGE | BufferUsage.COPY_SRC)
buffer_1 = device.create_buffer_with_data(data=data1, usage=BufferUsage.STORAGE | BufferUsage.COPY_SRC)

4. Le contrat des emplacements.

In [ ]:
bind_group_layout = device.create_bind_group_layout(entries=[
    {"binding": 0, "visibility": ShaderStage.COMPUTE, "buffer": {"type": BufferBindingType.storage}},
    {"binding": 1, "visibility": ShaderStage.COMPUTE, "buffer": {"type": BufferBindingType.storage}},
])

5. Les bind groups, un par sens du ping-pong.

In [ ]:
bind_group = device.create_bind_group(layout=bind_group_layout, entries=[
    {"binding": 0, "resource": {"buffer": buffer_0, "offset": 0, "size": buffer_0.size}},
    {"binding": 1, "resource": {"buffer": buffer_1, "offset": 0, "size": buffer_1.size}},
])